[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-05-document-loaders.ipynb#scrollTo=a1b2c3d4)

---
# Day 5 · Document Loaders — PDFs, Web Pages, and Directories
**certified-journeys / llm-engineering-certified** · Day 5 · Data Ingestion

> **Goal for today:** Load documents from PDFs, web pages, and directory trees using LangChain loaders, then enrich each Document's metadata so it's ready for a retrieval pipeline.


In [ ]:
%pip install -q langchain langchain-community pypdf beautifulsoup4 requests


## Step 1 · What is a Document Loader?

A **Document Loader** in LangChain converts raw sources (files, URLs, databases) into a list of `Document` objects. Each `Document` has two fields:

| Field | Type | Purpose |
|---|---|---|
| `page_content` | `str` | The actual text the LLM will read |
| `metadata` | `dict` | Source path, page numbers, timestamps, custom fields |

Loaders are the **first step** in every RAG pipeline. Good metadata here means good citations later.

Official concept page: https://python.langchain.com/docs/concepts/document_loaders/


In [ ]:
# Inspect the Document data class before loading anything
from langchain_core.documents import Document

sample = Document(
    page_content="LangChain makes it easy to build LLM applications.",
    metadata={"source": "manual", "author": "demo"}
)

print("page_content:", sample.page_content)
print("metadata    :", sample.metadata)
print("type        :", type(sample))


### What just happened?

- `Document` is a plain Pydantic model with exactly two fields — **no magic**.
- `metadata` is a free-form dict; loaders set `source` automatically, but you can add any keys you need.
- **`metadata['source']` is what RAG chains pass to citations** — keep it human-readable (a filename or URL, not a temp path).
- Downstream splitters and retrievers carry `metadata` forward unchanged, so enrichments you add here persist.


## Step 2 · Load a PDF with PyPDFLoader

`PyPDFLoader` splits a PDF into one `Document` per page. Each document's metadata includes `source` (file path) and `page` (0-indexed page number).

```
PyPDFLoader(path) → list[Document]
  └─ one Document per PDF page
  └─ metadata: {source, page}
```

Docs: https://python.langchain.com/docs/how_to/document_loader_pdf/


In [ ]:
import os, urllib.request
from langchain_community.document_loaders import PyPDFLoader

# Download a small public PDF for demo purposes
PDF_URL  = "https://www.w3.org/WAI/WCAG21/wcag21-diff.pdf"
PDF_PATH = "/tmp/sample.pdf"

if not os.path.exists(PDF_PATH):
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print("Downloaded PDF to", PDF_PATH)
else:
    print("PDF already on disk")

loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

print(f"\nTotal pages loaded : {len(pages)}")
print(f"Type of first item : {type(pages[0])}")
print(f"Metadata (page 0)  : {pages[0].metadata}")
print(f"Content preview    : {pages[0].page_content[:300]!r}")


### What just happened?

- `PyPDFLoader.load()` returns a `list[Document]` — one per PDF page.
- **`metadata['source']`** defaults to the file path; **`metadata['page']`** is the 0-based page index.
- `page_content` is plain text extracted by pypdf; tables and images are not preserved.
- Always inspect `pages[0]` immediately — blank pages or encoding issues show up early.
- **Production tip:** replace the local path with an S3/GCS URI by using `S3FileLoader` or a pre-signed URL download step.


## Step 3 · Load a Web Page with WebBaseLoader

`WebBaseLoader` fetches a URL and strips HTML tags via **BeautifulSoup**. You control which HTML elements to keep with `bs_kwargs`.

```
WebBaseLoader(url, bs_kwargs={"parse_only": SoupStrainer(...)})  
  └─ fetches the page with requests  
  └─ parses with bs4  
  └─ returns one Document (or one per URL)
```

Docs: https://python.langchain.com/docs/integrations/document_loaders/web_base/


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

# Only keep the main article body — ignore nav, footer, ads
bs_config = {"parse_only": SoupStrainer("article")}

loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs=bs_config,
)

web_docs = loader.load()

print(f"Documents returned : {len(web_docs)}")
print(f"Metadata           : {web_docs[0].metadata}")
print(f"Content length     : {len(web_docs[0].page_content)} chars")
print(f"First 400 chars    : {web_docs[0].page_content[:400]!r}")


### What just happened?

- `SoupStrainer("article")` tells BeautifulSoup to parse only `<article>` tags — this cuts noise dramatically.
- **`metadata['source']`** is the URL — exactly what a citation chain needs.
- Passing a list to `web_paths` lets you batch-load multiple URLs in one call.
- **If the page returns empty content**, the tag selector is wrong — inspect `soup.find("article")` in a scratch cell first.
- For JavaScript-rendered pages, swap `WebBaseLoader` for `AsyncChromiumLoader` (requires Playwright).


## Step 4 · Load a Directory of .txt Files with DirectoryLoader

`DirectoryLoader` walks a directory tree and applies a sub-loader to every matching file. Use `glob` to filter by extension.

```
DirectoryLoader(
    path      = "./docs/",
    glob      = "**/*.txt",     # recursive glob
    loader_cls = TextLoader,
)
```


In [ ]:
import os, pathlib
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Create a small local directory of .txt files for demo
DEMO_DIR = "/tmp/demo_docs"
pathlib.Path(DEMO_DIR).mkdir(parents=True, exist_ok=True)

sample_files = {
    "intro.txt"  : "LangChain is a framework for building LLM-powered applications.",
    "loaders.txt": "Document loaders convert raw files into Document objects with metadata.",
    "chains.txt" : "Chains connect prompts, models, and output parsers into a workflow.",
    "agents.txt" : "Agents use tools dynamically and decide which action to take next.",
}

for filename, content in sample_files.items():
    (pathlib.Path(DEMO_DIR) / filename).write_text(content)

# Load all .txt files recursively
loader = DirectoryLoader(
    DEMO_DIR,
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=False,
)

dir_docs = loader.load()

print(f"Files found: {len(dir_docs)}")
for doc in dir_docs:
    print(f"  source: {doc.metadata['source']!r}  |  {doc.page_content[:60]!r}")


### What just happened?

- `DirectoryLoader` delegates to `TextLoader` (or any loader class) for each matched file.
- **`glob="**/*.txt"`** is recursive — subdirectories are included.
- `metadata['source']` is the **full absolute path** by default — you'll want to normalize this (next step).
- Swap `TextLoader` for `PyPDFLoader`, `UnstructuredMarkdownLoader`, or any other loader to handle mixed-type directories.
- Use `silent_errors=True` to skip unreadable files instead of raising an exception.


## Step 5 · Enrich Document Metadata

After loading, add domain-specific fields to every Document so downstream components (splitters, vector stores, chains) have rich context:

| Field | Value | Used for |
|---|---|---|
| `source` | readable path or URL | Citations |
| `loaded_at` | ISO 8601 timestamp | Freshness filtering |
| `domain` | category tag | Filtered retrieval |

Metadata enrichment is done in a simple loop — no LangChain magic required.


In [ ]:
from datetime import datetime, timezone

def enrich_metadata(docs, domain: str) -> list:
    """Add loaded_at, domain, and normalise the source field."""
    now_iso = datetime.now(timezone.utc).isoformat()
    enriched = []
    for doc in docs:
        new_meta = dict(doc.metadata)          # copy so we don't mutate the original

        # Normalise source: strip leading /tmp/ paths to just filename
        raw_source = new_meta.get("source", "unknown")
        new_meta["source"]    = os.path.basename(raw_source) if "/tmp/" in raw_source else raw_source
        new_meta["loaded_at"] = now_iso
        new_meta["domain"]    = domain

        enriched.append(Document(page_content=doc.page_content, metadata=new_meta))
    return enriched

# Enrich the directory documents
enriched_docs = enrich_metadata(dir_docs, domain="langchain-basics")

# Inspect one enriched document
for doc in enriched_docs:
    print(doc.metadata)


### What just happened?

- Metadata enrichment is **just a loop** — copy the dict, update keys, wrap in a new `Document`.
- **`loaded_at`** enables freshness-based filtering in vector stores (`where loaded_at > '2024-01-01'`).
- **`domain`** enables multi-tenant or multi-topic retrieval — filter by domain before similarity search.
- Normalising `source` from an absolute path to a filename makes citations human-readable in the final answer.
- This enrichment step is also the right place to **add document IDs** for deduplication.


## Step 6 · Combine All Sources into a Single Document List

In a real RAG pipeline, you load from multiple sources and merge everything before splitting. Here we combine PDF pages, the web page, and directory files into one list.


In [ ]:
# Enrich all sources with consistent metadata
pdf_enriched = enrich_metadata(pages[:3], domain="wcag-spec")    # first 3 pages only
web_enriched = enrich_metadata(web_docs, domain="llm-agents-blog")
dir_enriched = enrich_metadata(dir_docs, domain="langchain-basics")

# Merge into one corpus
all_docs = pdf_enriched + web_enriched + dir_enriched

print(f"Total documents in corpus: {len(all_docs)}")
print("\nBreakdown by domain:")
from collections import Counter
for domain, count in Counter(d.metadata["domain"] for d in all_docs).items():
    print(f"  {domain:<25} {count} docs")

print("\nSample metadata entries:")
for doc in all_docs[:3]:
    print("  ", {k: v for k, v in doc.metadata.items() if k != "loaded_at"})


### What just happened?

- Merging is just Python list concatenation — all loaders return the same `Document` type.
- **`Counter` on `metadata['domain']`** gives an instant audit of what's in your corpus.
- This combined list is the input to a text splitter (Day 6) and then a vector store (Day 7).
- **Deduplication tip:** hash `page_content` and filter duplicates before ingesting — especially important for web crawls.


In [ ]:
# Challenge: Write a loader pipeline function
# Given a list of PDF paths, a list of URLs, and a directory path,
# load all three, enrich each with a 'pipeline_run' metadata field
# (set it to a unique run ID using uuid4), and return the merged list.
#
# Scaffold:
# import uuid
#
# def load_pipeline(pdf_paths: list[str], urls: list[str], dir_path: str) -> list:
#     run_id = str(uuid.uuid4())
#     all_docs = []
#
#     # 1. Load PDFs
#     # YOUR CODE HERE
#
#     # 2. Load web pages
#     # YOUR CODE HERE
#
#     # 3. Load directory
#     # YOUR CODE HERE
#
#     # 4. Enrich each document with pipeline_run=run_id and loaded_at
#     # YOUR CODE HERE
#
#     return all_docs
#
# result = load_pipeline(
#     pdf_paths=[PDF_PATH],
#     urls=["https://python.langchain.com/docs/concepts/document_loaders/"],
#     dir_path=DEMO_DIR,
# )
# print(f"Loaded {len(result)} documents with run_id in metadata")


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `Document` | Plain Pydantic model: `page_content` + `metadata` dict |
| `PyPDFLoader` | One Document per PDF page; `metadata['page']` is 0-indexed |
| `WebBaseLoader` | Use `SoupStrainer` to target the right HTML element — no strainer = noisy text |
| `DirectoryLoader` | `glob="**/*.txt"` is recursive; set `loader_cls` to match file type |
| Metadata enrichment | Always set `source` (readable), `loaded_at`, and domain tags before splitting |
| `metadata['source']` | This field becomes citation text — keep it human-readable |

> **Tip:** Always inspect a sample Document after loading — metadata.source is what gets passed to citations, so make it readable.

---
## What's next
**Day 6** → Text Splitting Strategies — learn how to chunk your loaded Documents into sizes that fit context windows without losing meaning.

Mark Day 5 complete in your [tracker](../index.html).
